# Fresh Recovery Validation

This notebook verifies owner-side recovery of the pinned QLoRA adapter and reproduces the fixed 300-sample evaluation. It checks out the exact repository commit used by the recorded validation.

The adapter repository is private, so an authorized Kaggle `HF_TOKEN` secret is required. This is a reproducibility and artifact-integrity check, not a clinical validation.


In [ ]:
# ============================================================
# Cell 1 — Runtime Configuration
# ============================================================

import os
from pathlib import Path

# ------------------------------------------------------------
# 1. Force single-GPU execution
# Must be set BEFORE importing torch / unsloth
# ------------------------------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ------------------------------------------------------------
# 2. GitHub project
# ------------------------------------------------------------
REPO_URL = "https://github.com/Leng-Bu-Ding/medical-llm-qlora.git"
PROJECT_DIR = Path("/kaggle/working/medical-llm-qlora")

# ------------------------------------------------------------
# 3. Hugging Face QLoRA Adapter
# ------------------------------------------------------------
ADAPTER_REPO = "Lengbuding/llama3-medquad-qlora"

ADAPTER_REVISION = (
    "d26749288d78cab0839468dfa532a3beafcba871"
)

ADAPTER_DIR = Path("/kaggle/working/restored_adapter")

# ------------------------------------------------------------
# 4. Immutable Base Model revision
# ------------------------------------------------------------
EXPECTED_BASE_REVISION = (
    "fd5a4dc328319c1cfe9489eccfb9c6406bdfd469"
)

# ------------------------------------------------------------
# 5. Fresh Recovery outputs
# ------------------------------------------------------------
RUN_DIR = Path("/kaggle/working/fresh_recovery")

DATA_DIR = RUN_DIR / "data"
PREDICTIONS_PATH = RUN_DIR / "predictions_300.jsonl"
EVALUATION_PATH = RUN_DIR / "evaluation_summary.json"

RUN_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 6. Expected evaluation dataset
# ------------------------------------------------------------
EXPECTED_EVAL_COUNT = 300

EXPECTED_EVAL_SHA256 = (
    "281926cc8d8b7eb38edeac3e3882617be38a0658bba1fab37d90575a5a4165f8"
)

# ------------------------------------------------------------
# 7. Original clean_main_v1 metrics
# Used only for final comparison
# ------------------------------------------------------------
ORIGINAL_METRICS = {
    "base_bertscore_f1": 0.58587105,
    "ft_bertscore_f1": 0.68233270,
    "base_rougeL": 0.19025864,
    "ft_rougeL": 0.33346714,
    "improved": 247,
    "regressed": 53,
}

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("===== Fresh Recovery Configuration =====")
print("Visible GPU          :", os.environ["CUDA_VISIBLE_DEVICES"])
print("Project directory    :", PROJECT_DIR)
print("Adapter repository   :", ADAPTER_REPO)
print("Adapter revision     :", ADAPTER_REVISION)
print("Base revision        :", EXPECTED_BASE_REVISION)
print("Run directory        :", RUN_DIR)
print("Expected eval count  :", EXPECTED_EVAL_COUNT)
print("Expected eval SHA256 :", EXPECTED_EVAL_SHA256)

print("\n✅ Runtime configuration initialized.")

In [ ]:
# ============================================================
# Cell 2 — Project Bootstrap
# Clone GitHub + install dependencies
# ============================================================

import os
import subprocess
from pathlib import Path

EXPECTED_GIT_COMMIT = "c8056f8fdf0a9a2d059fdda277d89d68516bc79a"

# ------------------------------------------------------------
# 1. Clone repository
# ------------------------------------------------------------
if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    print("Project already exists, skip clone.")

os.chdir(PROJECT_DIR)

# ------------------------------------------------------------
# 2. Check out and verify the exact code used for recovery
# ------------------------------------------------------------
subprocess.run(
    ["git", "checkout", "--detach", EXPECTED_GIT_COMMIT],
    check=True,
)

# ------------------------------------------------------------
# 3. Verify Git commit
# ------------------------------------------------------------
git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

print("\n===== Git Repository =====")
print("Project directory :", PROJECT_DIR)
print("Git commit        :", git_commit)

assert git_commit == EXPECTED_GIT_COMMIT, (
    f"Unexpected Git commit: {git_commit}"
)

# ------------------------------------------------------------
# 4. Install dependencies
# ------------------------------------------------------------
print("\n===== Installing Dependencies =====")

subprocess.run(
    [
        "python", "-m", "pip", "install",
        "-q",
        "-r", "requirements-train.txt",
    ],
    check=True,
)

subprocess.run(
    [
        "python", "-m", "pip", "install",
        "-q",
        "-e", ".",
        "--no-deps",
    ],
    check=True,
)

# ------------------------------------------------------------
# 5. Basic file checks
# ------------------------------------------------------------
required_paths = [
    PROJECT_DIR / "configs" / "qlora_llama3_8b.yaml",
    PROJECT_DIR / "src" / "medical_llm",
    PROJECT_DIR / "scripts" / "run_inference.py",
]

print("\n===== Project Check =====")

for path in required_paths:
    print(f"{path.relative_to(PROJECT_DIR)} :", path.exists())
    assert path.exists()

print("\n✅ GitHub project and Python environment restored.")

In [ ]:
# ============================================================
# Cell 3 — Hugging Face Authentication
# ============================================================

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

# ------------------------------------------------------------
# 1. Read HF token from Kaggle Secret
# ------------------------------------------------------------
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

assert hf_token is not None
assert len(hf_token) > 0

# ------------------------------------------------------------
# 2. Verify Hugging Face authentication
# ------------------------------------------------------------
api = HfApi(token=hf_token)

user_info = api.whoami()

print("===== Hugging Face Authentication =====")
print("User :", user_info["name"])

assert user_info["name"] == "Lengbuding"

print("\n✅ Hugging Face authentication successful.")

In [ ]:
# ============================================================
# Cell 4 — Restore QLoRA Adapter
# ============================================================

from huggingface_hub import snapshot_download
from pathlib import Path

# ------------------------------------------------------------
# 1. Verify private adapter repository
# ------------------------------------------------------------
repo_info = api.model_info(ADAPTER_REPO)

print("===== Adapter Repository =====")
print("Repository :", repo_info.id)
print("Private    :", repo_info.private)
print("Revision   :", repo_info.sha)

assert repo_info.id == ADAPTER_REPO
assert repo_info.private is True

# ------------------------------------------------------------
# 2. Download exact adapter revision
# ------------------------------------------------------------
snapshot_download(
    repo_id=ADAPTER_REPO,
    revision=ADAPTER_REVISION,
    local_dir=str(ADAPTER_DIR),
    token=hf_token,
)

# ------------------------------------------------------------
# 3. Verify required files
# ------------------------------------------------------------
required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]

print("\n===== Restored Adapter Files =====")

for name in required_files:
    path = ADAPTER_DIR / name
    print(f"{name:<28} :", path.exists())
    assert path.exists(), f"Missing required file: {name}"

adapter_size_mb = (
    (ADAPTER_DIR / "adapter_model.safetensors").stat().st_size
    / 1024**2
)

print(f"\nAdapter size : {adapter_size_mb:.2f} MB")
print("Adapter path :", ADAPTER_DIR)

print("\n✅ Exact QLoRA Adapter restored.")

In [ ]:
# ============================================================
# Cell 5 — Adapter / Base Revision Integrity Check
# ============================================================

import json
import yaml
import hashlib

# ------------------------------------------------------------
# 1. Read GitHub project config
# ------------------------------------------------------------
CONFIG_PATH = PROJECT_DIR / "configs" / "qlora_llama3_8b.yaml"

with open(CONFIG_PATH, encoding="utf-8") as f:
    project_config = yaml.safe_load(f)

# ------------------------------------------------------------
# 2. Read Adapter config
# ------------------------------------------------------------
ADAPTER_CONFIG_PATH = ADAPTER_DIR / "adapter_config.json"

with open(ADAPTER_CONFIG_PATH, encoding="utf-8") as f:
    adapter_config = json.load(f)

project_base_model = project_config["model"]["name"]
project_base_revision = project_config["model"]["revision"]

adapter_base_model = adapter_config["base_model_name_or_path"]
adapter_base_revision = adapter_config.get("revision")

# ------------------------------------------------------------
# 3. Verify model identity
# ------------------------------------------------------------
print("===== Base Model Integrity =====")
print("Project Base Model  :", project_base_model)
print("Adapter Base Model  :", adapter_base_model)

print("\nProject Revision     :", project_base_revision)
print("Adapter Revision     :", adapter_base_revision)
print("Expected Revision    :", EXPECTED_BASE_REVISION)

assert project_base_model == adapter_base_model, \
    "Base model mismatch!"

assert project_base_revision == EXPECTED_BASE_REVISION, \
    "GitHub config base revision mismatch!"

assert adapter_base_revision == EXPECTED_BASE_REVISION, \
    "Adapter base revision mismatch!"

# ------------------------------------------------------------
# 4. Record Adapter SHA256
# ------------------------------------------------------------
adapter_weight_path = ADAPTER_DIR / "adapter_model.safetensors"

sha256 = hashlib.sha256()

with open(adapter_weight_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

ADAPTER_SHA256 = sha256.hexdigest()

print("\n===== Adapter Artifact =====")
print("LoRA rank            :", adapter_config["r"])
print("LoRA alpha           :", adapter_config["lora_alpha"])
print("LoRA dropout         :", adapter_config["lora_dropout"])
print("Adapter SHA256       :", ADAPTER_SHA256)

print("\n✅ Adapter and immutable Base Model revision are consistent.")

In [ ]:
# ============================================================
# Cell 6 — Load Recovered Base Model + QLoRA Adapter
# ============================================================

import sys
import torch

# ------------------------------------------------------------
# 1. Make project package importable
# ------------------------------------------------------------
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from medical_llm.config import load_config
from medical_llm.inference import load_finetuned_model

# ------------------------------------------------------------
# 2. Confirm single-GPU environment
# ------------------------------------------------------------
print("===== GPU Environment =====")
print("CUDA available :", torch.cuda.is_available())
print("GPU count      :", torch.cuda.device_count())

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1, \
    f"Expected exactly 1 visible GPU, got {torch.cuda.device_count()}"

print("GPU            :", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 3. Load project config
# ------------------------------------------------------------
config = load_config(str(CONFIG_PATH))

# ------------------------------------------------------------
# 4. Restore Base Model + QLoRA Adapter
# ------------------------------------------------------------
print("\n===== Loading Recovered Model =====")

model, tokenizer = load_finetuned_model(
    config=config,
    adapter_path=ADAPTER_DIR,
)

# ------------------------------------------------------------
# 5. Verify PEFT model
# ------------------------------------------------------------
print("\n===== Recovery Check =====")
print("Model class     :", model.__class__.__name__)
print("Tokenizer class :", tokenizer.__class__.__name__)
print("Has PEFT config :", hasattr(model, "peft_config"))

vram_gib = torch.cuda.memory_allocated(0) / 1024**3
print(f"VRAM allocated  : {vram_gib:.2f} GiB")

assert hasattr(model, "peft_config"), \
    "Recovered model does not contain PEFT/LoRA configuration."

print("\n✅ Base Model + QLoRA Adapter restored successfully.")

In [ ]:
# ============================================================
# Cell 7 — Smoke Test: Base vs Fine-tuned
# Purpose:
#   Verify that the recovered model can generate normally
#   and that the restored LoRA adapter actually affects behavior.
# ============================================================

import torch

SMOKE_QUESTION = "What are the common symptoms of type 2 diabetes?"

# ------------------------------------------------------------
# 1. Build exactly one prompt
# ------------------------------------------------------------
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": SMOKE_QUESTION}],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda")

generation_kwargs = {
    "max_new_tokens": config["generation"]["max_new_tokens"],
    "do_sample": False,
    "use_cache": True,
    "pad_token_id": tokenizer.eos_token_id,
}

# ------------------------------------------------------------
# 2. Small reusable generation helper
# ------------------------------------------------------------
def generate_one():
    with torch.inference_mode():
        outputs = model.generate(  # noqa: F821 -- assigned in the previous cell
            **inputs,
            **generation_kwargs,
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(  # noqa: F821 -- assigned in the previous cell
        new_tokens,
        skip_special_tokens=True,
    ).strip()

# ------------------------------------------------------------
# 3. Base Model — disable LoRA
# ------------------------------------------------------------
with model.disable_adapter():
    base_answer = generate_one()

# ------------------------------------------------------------
# 4. Fine-tuned Model — LoRA enabled
# ------------------------------------------------------------
ft_answer = generate_one()

# ------------------------------------------------------------
# 5. Verify outputs
# ------------------------------------------------------------
print("===== Question =====")
print(SMOKE_QUESTION)

print("\n===== Base Model =====")
print(base_answer)

print("\n===== Fine-tuned Model =====")
print(ft_answer)

print("\n===== Smoke Test =====")
print("Base output non-empty :", len(base_answer) > 0)
print("FT output non-empty   :", len(ft_answer) > 0)
print("Outputs identical     :", base_answer == ft_answer)

assert len(base_answer) > 0
assert len(ft_answer) > 0

print("\n✅ Recovered model inference works.")
print("✅ Base / Fine-tuned comparison completed.")

In [ ]:
# ============================================================
# Cell 8 — Reproduce Exact 300-Sample Evaluation Set
# ============================================================

from medical_llm.prepare import prepare_medquad

# ------------------------------------------------------------
# 1. Rebuild MedQuAD clean split
#    Reuse the tokenizer already loaded in Cell 6
# ------------------------------------------------------------
print("===== Rebuilding Evaluation Dataset =====")

data_manifest = prepare_medquad(
    config=config,
    tokenizer=tokenizer,
    output_dir=DATA_DIR,
    protocol="clean",
)

# ------------------------------------------------------------
# 2. Read evaluation_300 information
# ------------------------------------------------------------
eval_info = data_manifest["files"]["evaluation_300"]

fresh_eval_count = eval_info["count"]
fresh_eval_sha256 = eval_info["sha256"]

# ------------------------------------------------------------
# 3. Compare against original clean_main_v1
# ------------------------------------------------------------
print("\n===== Evaluation Dataset Integrity =====")
print("Fresh count      :", fresh_eval_count)
print("Expected count   :", EXPECTED_EVAL_COUNT)

print("\nFresh SHA256     :", fresh_eval_sha256)
print("Expected SHA256  :", EXPECTED_EVAL_SHA256)

count_match = fresh_eval_count == EXPECTED_EVAL_COUNT
hash_match = fresh_eval_sha256 == EXPECTED_EVAL_SHA256

print("\nCount match      :", count_match)
print("SHA256 match     :", hash_match)

assert count_match, "Evaluation sample count mismatch!"
assert hash_match, "Evaluation dataset SHA256 mismatch!"

EVAL_DATA_PATH = DATA_DIR / "evaluation_300.jsonl"

assert EVAL_DATA_PATH.exists()

print("\nEvaluation file  :", EVAL_DATA_PATH)
print("\n✅ Exact 300-sample evaluation dataset reproduced.")

In [ ]:
# ============================================================
# Cell 9 — 300-Sample Paired Inference
# Base Model vs Fine-tuned Model
# ============================================================

import json
import hashlib
import torch

from tqdm.auto import tqdm

from medical_llm.data import read_jsonl
from medical_llm.inference import _generate_batch, prompt_hash


# ------------------------------------------------------------
# 1. Load the exact 300 evaluation samples
# ------------------------------------------------------------
examples = read_jsonl(EVAL_DATA_PATH)

assert len(examples) == 300

print("===== Paired Inference Setup =====")
print("Evaluation samples :", len(examples))
print("Batch size         :", config["generation"]["batch_size"])
print("Max new tokens     :", config["generation"]["max_new_tokens"])
print("Do sample          :", config["generation"]["do_sample"])
print("Output             :", PREDICTIONS_PATH)


# ------------------------------------------------------------
# 2. Reproduce generation configuration hash
# ------------------------------------------------------------
generation_config = dict(config["generation"])

generation_hash = hashlib.sha256(
    json.dumps(
        generation_config,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

EXPECTED_GENERATION_HASH = (
    "b11c4bf12c11bb2adab32621352365dfb6c10bb6e812a1146a86edf15102a593"
)

print("\nGeneration hash     :", generation_hash)
print("Expected hash       :", EXPECTED_GENERATION_HASH)
print("Generation match    :", generation_hash == EXPECTED_GENERATION_HASH)

assert generation_hash == EXPECTED_GENERATION_HASH


# ------------------------------------------------------------
# 3. Run paired Base / Fine-tuned inference
# ------------------------------------------------------------
batch_size = int(config["generation"]["batch_size"])

PREDICTIONS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

count = 0

with PREDICTIONS_PATH.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as stream:

    for start in tqdm(
        range(0, len(examples), batch_size),
        desc="Paired inference",
    ):

        batch = examples[start:start + batch_size]
        questions = [item.question for item in batch]

        # -------------------------
        # Base Model
        # LoRA temporarily disabled
        # -------------------------
        with torch.inference_mode():
            with model.disable_adapter():
                base_outputs = _generate_batch(
                    model,
                    tokenizer,
                    questions,
                    config,
                )

        # -------------------------
        # Fine-tuned Model
        # LoRA enabled
        # -------------------------
        with torch.inference_mode():
            ft_outputs = _generate_batch(
                model,
                tokenizer,
                questions,
                config,
            )

        # -------------------------
        # Save paired predictions
        # -------------------------
        for item, base_text, ft_text in zip(
            batch,
            base_outputs,
            ft_outputs,
            strict=True,
        ):
            record = {
                **item.to_dict(),

                "prompt_hash": prompt_hash(
                    tokenizer,
                    item.question,
                ),

                "generation_hash": generation_hash,

                "base_prediction": base_text.strip(),

                "ft_prediction": ft_text.strip(),

                "generation": generation_config,

                "model_revision": config["model"]["revision"],
            }

            stream.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True,
                )
                + "\n"
            )

            count += 1

        # Save progress continuously
        stream.flush()


# ------------------------------------------------------------
# 4. Integrity checks
# ------------------------------------------------------------
with PREDICTIONS_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    recovered_predictions = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

sample_ids = [
    row["sample_id"]
    for row in recovered_predictions
]

print("\n===== Paired Inference Result =====")
print("Generated records :", len(recovered_predictions))
print("Unique sample IDs :", len(set(sample_ids)))
print("Output file       :", PREDICTIONS_PATH)

assert count == 300
assert len(recovered_predictions) == 300
assert len(set(sample_ids)) == 300

print("\n✅ 300 Base + 300 Fine-tuned predictions generated successfully.")

In [ ]:
# ============================================================
# Cell 10 — Evaluate Fresh Recovery Predictions
# ROUGE + BERTScore + Bootstrap CI + Paired Comparison
# ============================================================

import gc
import json
import torch

from medical_llm.evaluation import (
    read_prediction_jsonl,
    evaluate_predictions,
)

# ------------------------------------------------------------
# 1. Release Llama-3 from GPU
# BERTScore needs GPU memory for its encoder model
# ------------------------------------------------------------
print("===== Releasing Llama Model =====")

del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

vram_after_release = torch.cuda.memory_allocated(0) / 1024**3

print(f"VRAM allocated after release : {vram_after_release:.2f} GiB")


# ------------------------------------------------------------
# 2. Load the 300 paired predictions
# ------------------------------------------------------------
records = read_prediction_jsonl(PREDICTIONS_PATH)

print("\n===== Evaluation Setup =====")
print("Prediction records :", len(records))
print("BERTScore model    :", config["evaluation"]["bertscore_model"])
print("Bootstrap samples  :", config["evaluation"]["bootstrap_samples"])
print("Confidence level   :", config["evaluation"]["confidence_level"])

assert len(records) == 300


# ------------------------------------------------------------
# 3. Run the project's original evaluation pipeline
# ------------------------------------------------------------
print("\n===== Running Evaluation =====")
print("This may take several minutes because BERTScore loads a large encoder.\n")

report = evaluate_predictions(
    records,
    config,
)

# Separate error cases, matching the project's original output structure
error_cases = report.pop("error_cases")


# ------------------------------------------------------------
# 4. Save evaluation artifacts
# ------------------------------------------------------------
EVALUATION_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

ERROR_CASES_PATH = RUN_DIR / "error_cases.jsonl"

with ERROR_CASES_PATH.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as f:
    for item in error_cases:
        f.write(
            json.dumps(
                item,
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )


# ------------------------------------------------------------
# 5. Print core Fresh Recovery metrics
# ------------------------------------------------------------
metrics = report["metrics"]

base = metrics["base_prediction"]
ft = metrics["ft_prediction"]
delta = metrics["paired_delta"]
comparison = metrics["comparison"]

print("\n===== Fresh Recovery Metrics =====")

print(
    f"Base BERTScore F1 : {base['bertscore_f1']['mean']:.8f}"
)
print(
    f"FT   BERTScore F1 : {ft['bertscore_f1']['mean']:.8f}"
)
print(
    f"Δ    BERTScore F1 : {delta['bertscore_f1']['mean_delta']:+.8f}"
)

print()

print(
    f"Base ROUGE-1      : {base['rouge1']['mean']:.8f}"
)
print(
    f"FT   ROUGE-1      : {ft['rouge1']['mean']:.8f}"
)

print(
    f"Base ROUGE-2      : {base['rouge2']['mean']:.8f}"
)
print(
    f"FT   ROUGE-2      : {ft['rouge2']['mean']:.8f}"
)

print(
    f"Base ROUGE-L      : {base['rougeL']['mean']:.8f}"
)
print(
    f"FT   ROUGE-L      : {ft['rougeL']['mean']:.8f}"
)
print(
    f"Δ    ROUGE-L      : {delta['rougeL']['mean_delta']:+.8f}"
)

print()

print("Improved           :", comparison["improved"])
print("Regressed          :", comparison["regressed"])
print("Tied               :", comparison["tied"])

print("\nEvaluation summary :", EVALUATION_PATH)
print("Error cases        :", ERROR_CASES_PATH)

assert report["sample_count"] == 300

print("\n✅ Fresh Recovery evaluation completed.")

In [ ]:
# ============================================================
# Cell 11 — Original vs Fresh Recovery Comparison
# ============================================================

# ------------------------------------------------------------
# 1. Original clean_main_v1 metrics
# ------------------------------------------------------------
original = {
    "Base BERTScore F1": 0.58587105,
    "FT BERTScore F1":   0.68233270,
    "Δ BERTScore F1":    0.09646165,

    "Base ROUGE-1":      0.32064814,
    "FT ROUGE-1":        0.42869113,

    "Base ROUGE-2":      0.09670771,
    "FT ROUGE-2":        0.24860901,

    "Base ROUGE-L":      0.19025864,
    "FT ROUGE-L":        0.33346714,
    "Δ ROUGE-L":         0.14320850,
}

# ------------------------------------------------------------
# 2. Fresh Recovery metrics
# ------------------------------------------------------------
fresh = {
    "Base BERTScore F1": base["bertscore_f1"]["mean"],
    "FT BERTScore F1":   ft["bertscore_f1"]["mean"],
    "Δ BERTScore F1":    delta["bertscore_f1"]["mean_delta"],

    "Base ROUGE-1":      base["rouge1"]["mean"],
    "FT ROUGE-1":        ft["rouge1"]["mean"],

    "Base ROUGE-2":      base["rouge2"]["mean"],
    "FT ROUGE-2":        ft["rouge2"]["mean"],

    "Base ROUGE-L":      base["rougeL"]["mean"],
    "FT ROUGE-L":        ft["rougeL"]["mean"],
    "Δ ROUGE-L":         delta["rougeL"]["mean_delta"],
}

# ------------------------------------------------------------
# 3. Compare
# ------------------------------------------------------------
TOLERANCE = 1e-7

print("===== Original vs Fresh Recovery =====")
print(
    f"{'Metric':<22}"
    f"{'Original':>14}"
    f"{'Fresh':>14}"
    f"{'Difference':>16}"
    f"{'Status':>10}"
)

print("-" * 76)

all_metrics_pass = True

for metric in original:
    old = original[metric]
    new = fresh[metric]
    diff = new - old

    passed = abs(diff) <= TOLERANCE
    all_metrics_pass &= passed

    print(
        f"{metric:<22}"
        f"{old:>14.8f}"
        f"{new:>14.8f}"
        f"{diff:>+16.10f}"
        f"{'PASS' if passed else 'FAIL':>10}"
    )

# ------------------------------------------------------------
# 4. Compare discrete results
# ------------------------------------------------------------
count_checks = {
    "Improved": (247, comparison["improved"]),
    "Regressed": (53, comparison["regressed"]),
    "Tied": (0, comparison["tied"]),
}

print("\n===== Sample-Level Comparison =====")

all_counts_pass = True

for name, (old, new) in count_checks.items():
    passed = old == new
    all_counts_pass &= passed

    print(
        f"{name:<10}: "
        f"Original={old:<3}  "
        f"Fresh={new:<3}  "
        f"{'PASS' if passed else 'FAIL'}"
    )

# ------------------------------------------------------------
# 5. Final verdict
# ------------------------------------------------------------
recovery_pass = all_metrics_pass and all_counts_pass

print("\n===== Final Verdict =====")

if recovery_pass:
    print("✅ PASS — Fresh Recovery reproduces clean_main_v1.")
    print(f"Metric tolerance: {TOLERANCE}")
else:
    print("❌ FAIL — Fresh Recovery differs from clean_main_v1.")

assert recovery_pass

print("\n✅ Numerical reproducibility verified.")